In [1]:
import dspy

## Quickstart

In [20]:
# 1️⃣ Configurar el modelo local de Ollama
lm = dspy.LM(
    model="lm_studio/medgemma-4b-it",
    api_base="http://localhost:1234/v1",
)

dspy.configure(lm=lm)

La descripción puesta en el docstring es el System Prompt

In [ ]:
# 2️⃣ Definir la tarea (Signature)
class WeatherClassifier(dspy.Signature):
    """
    You are a weather expert.
    Classify the weather condition based on a short description.
    Possible outputs: "Sunny", "Rainy", or "Cloudy".
    """
    description = dspy.InputField(desc="Short text describing the weather")
    category = dspy.OutputField(desc="Sunny, Rainy, or Cloudy")

In [22]:
# 3️⃣ Crear el predictor
classifier = dspy.Predict(WeatherClassifier)

In [23]:
# 4️⃣ Probar con ejemplos simples
examples = [
    "The sun is shining and the sky is clear.",
    "Dark clouds are gathering and it starts to rain.",
    "The sky is grey and there is no sunlight.",
]

print("⚙️ Clasificación directa:\n")
for text in examples:
    result = classifier(description=text)
    print(f"🌦️  '{text}' → 🧠 {result.category}")

⚙️ Clasificación directa:

🌦️  'The sun is shining and the sky is clear.' → 🧠 Sunny
🌦️  'Dark clouds are gathering and it starts to rain.' → 🧠 Rainy
🌦️  'The sky is grey and there is no sunlight.' → 🧠 Cloudy


## Ejemplo con Tarea 1

In [79]:
import json
import re
# Esto toma los ejemplos ICL y los formatea automáticamente de la mejor forma para el modelo
from dspy.teleprompt import LabeledFewShot
from dspy.teleprompt import LabeledFewShot

In [ ]:
# Función de conteo de picos
def count_peaks(seq: str) -> int:
    """Cuenta los picos '|' y ':' en una secuencia."""
    return seq.count('|') + seq.count(':')

# Función de parsing
def extract_json(text: str):
    """Extrae el JSON desde texto con formato irregular."""
    if not text:
        return None
    # Buscar bloque JSON
    match = re.search(r'\{[\s\S]*\}', text)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            return None
    return None

# Definir la tarea (Signature)
class ECGClassifier(dspy.Signature):
    """
    You are an expert in analyzing symbolic ECG sequences composed of '.', '|', ':', '_', and '~'.

    Each '|' or ':' counts as one peak.
    total_peaks = count('|') + count(':')

    Classify the label for the ECG based on total_peaks:
        - Brady: 4–7 peaks (<60 bpm)
        - Normal: 8–11 peaks (60–100 bpm)
        - Tachy: 12–18 peaks (>100 bpm)
    """
    # Datos de entrada
    sequence = dspy.InputField(desc="Symbolic ECG sequence (. | : _ ~)")
    # Campos de salida
    total_peaks = dspy.OutputField(desc="Total number of peaks in the ECG sequence")
    classification = dspy.OutputField(desc="Label: Brady, Normal, or Tachy")

In [ ]:
# Configurar modelo local
lm = dspy.LM(
    model="lm_studio/medgemma-4b-it",
    api_base="http://localhost:1234/v1",
    temperature=0.1,
)
dspy.configure(lm=lm)

# Ejemplos de prueba de cada clase
# Hay alguna manera de explicar por qué el ouput es el realmente el que es
train_examples = [
    dspy.Example(sequence=".|:|:|:|", classification="Brady").with_inputs('sequence'),
    dspy.Example(sequence=".|:|:|:|:|:", classification="Normal").with_inputs('sequence'),
    dspy.Example(sequence=".|:|:|:|:|:|:|:", classification="Tachy").with_inputs('sequence'),
]

# Ejemplo a predecir
test_samples = {
    "Brady": ".|:|:|:|",               # 7 picos
    "Normal": ".|:|:|:|:|:",          # 10 picos
    "Tachy": ".|:|:|:|:|:|:|:",       # 14 picos
}

In [86]:
def get_results(test_samples, classifier):
    for label, input_seq in test_samples.items():
        true_peaks = count_peaks(input_seq)
        if 4 <= true_peaks <= 7:
            expected_class = "Brady"
        elif 8 <= true_peaks <= 11:
            expected_class = "Normal"
        elif 12 <= true_peaks <= 18:
            expected_class = "Tachy"
        else:
            expected_class = "OutOfRange"

        result = classifier(sequence=input_seq)
        #print(result)
        predicted_peaks = int(result.total_peaks)
        predicted_class = result.classification

        correct = predicted_class.lower() == expected_class.lower()
        mark = "✅" if correct else "❌"
        comment = "Nº picos correcto ✅ " if true_peaks == predicted_peaks else "Nº de picos incorrecto ❌"
        
        print(f"🫀 {label:<10} | Picos reales: {true_peaks:<2} | Esperado: {expected_class:<7} "
            f"| Predicción → {predicted_class:<7} ({predicted_peaks:<2}) {mark} {comment}")

In [87]:
# Crear el predictor
teleprompter = LabeledFewShot(k=3)
classifier = teleprompter.compile(student=dspy.Predict(ECGClassifier), trainset=train_examples)

get_results(test_samples, classifier)

🫀 Brady      | Picos reales: 7  | Esperado: Brady   | Predicción → Brady   (4 ) ✅ Nº de picos incorrecto ❌
🫀 Normal     | Picos reales: 10 | Esperado: Normal  | Predicción → Normal  (10) ✅ Nº picos correcto ✅ 
🫀 Tachy      | Picos reales: 14 | Esperado: Tachy   | Predicción → Tachy   (14) ✅ Nº picos correcto ✅ 


Añadiendo chain of thought

In [85]:
classifier = teleprompter.compile(student=dspy.ChainOfThought(ECGClassifier), trainset=train_examples)
get_results(test_samples, classifier)

Prediction(
    reasoning='Not supplied for this particular example.',
    total_peaks='Not supplied for this particular example.',
    classification='Brady'
)


ValueError: invalid literal for int() with base 10: 'Not supplied for this particular example.'

In [88]:
lm.inspect_history(n=1)





[2025-12-18T12:35:51.354989]

System message:

Your input fields are:
1. `sequence` (str): Symbolic ECG sequence (. | : _ ~)
Your output fields are:
1. `total_peaks` (str): Total number of peaks in the ECG sequence
2. `classification` (str): ECG classification: Brady, Normal, or Tachy
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## sequence ## ]]
{sequence}

[[ ## total_peaks ## ]]
{total_peaks}

[[ ## classification ## ]]
{classification}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You are an expert in analyzing symbolic ECG sequences composed of '.', '|', ':', '_', and '~'.
        
        Each '|' or ':' counts as one peak.
        total_peaks = count('|') + count(':')
        
        Classify the label for the ECG based on total_peaks:
            - Brady: 4–7 peaks (<60 bpm)
            - Normal: 8–11 peaks (60–100 bpm)
            - Tachy: 12–18 peaks (>100 bpm)
        
        Ex